# 02 - Leaf gas exchange and its environmental drivers

The results generated here were used in manuscript section 3.3. Two questions:

1. **Is there secondary sexual dimorphism in leaf function?** Female and male
   means are compared within each cultivation system (Table 2).
2. **What drives each ecophysiological trait?** A gradient boosting machine
   predicts each trait from the environmental conditions of its measurement
   campaign, and permutation importance ranks the drivers (Figure 4).

Gas exchange was measured only at midday, so the light drivers offered to the
models are midday-only. Apparent quantum yield (Phi) is excluded as a
regression target because it *is* a light response by definition - it is the
slope of A against PPFD, so it is compared between sexes by ANCOVA instead.

**Produces:** Table 2, Figure 4.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

from yerbamate import config as C
from yerbamate import io_physio, models as M, plotting as P, stats as S

P.use_paper_style()
pd.set_option("display.width", 170, "display.max_columns", 30)

physio = pd.read_csv(C.PHYSIO_CLEAN)
growth = pd.read_csv(C.UNIFIED)
print(f"{len(physio):,} leaf measurements over {physio.period_label.nunique()} campaigns")
print(physio.groupby(["environment", "Sexo"]).size().to_string())

## Table 2 - sexual dimorphism in leaf gas exchange

Seven traits compared by t-test within each system; Phi compared by ANCOVA on
the A-vs-PPFD slopes. Females lead in monoculture for A, E and - most clearly -
intrinsic water-use efficiency; in agroforestry the female advantage shifts to
A, LUE and leaf-to-air temperature difference.

In [ ]:
rows = []
aqy = {}
for env in ["MO", "FUS"]:
    sub = physio[physio.environment == env]
    for v in C.PHYSIO_VARS:
        f = sub[sub.Sexo == "F"][v].dropna()
        m = sub[sub.Sexo == "M"][v].dropna()
        t, p = ttest_ind(f, m)
        pooled = ((len(f) - 1) * f.std() ** 2 + (len(m) - 1) * m.std() ** 2) / (len(f) + len(m) - 2)
        rows.append({"System": io_physio.SYSTEM_LABEL[env], "Trait": v,
                     "FE_mean": round(f.mean(), 4), "FE_sd": round(f.std(), 4),
                     "MA_mean": round(m.mean(), 4), "MA_sd": round(m.std(), 4),
                     "n_FE": len(f), "n_MA": len(m),
                     "t": round(t, 3), "p": round(p, 5), "sig": S.stars(p),
                     "Cohen_d": round((f.mean() - m.mean()) / np.sqrt(pooled), 3),
                     "test": "t-test"})
    a = io_physio.aqy_by_sex(sub)
    aqy[env] = a
    rows.append({"System": io_physio.SYSTEM_LABEL[env], "Trait": "Phi",
                 "FE_mean": round(a["F"], 5), "FE_sd": np.nan,
                 "MA_mean": round(a["M"], 5), "MA_sd": np.nan,
                 "n_FE": a["n_F"], "n_MA": a["n_M"],
                 "t": np.nan, "p": round(a["p"], 5), "sig": S.stars(a["p"]),
                 "Cohen_d": np.nan, "test": "ANCOVA (slope difference)"})

table_2 = pd.DataFrame(rows)
table_2.to_csv(C.TAB_DIR / "Table_2_gas_exchange_dimorphism.csv", index=False)
print(table_2[["System", "Trait", "FE_mean", "MA_mean", "p", "sig", "test"]].to_string(index=False))

## Figure 4 - environmental drivers of each ecophysiological trait

One GBM per trait, trained on the nine midday environmental features of the
measurement campaign. Bars are permutation importance as a percentage; the box
gives the 5-fold cross-validated R-squared.

The pattern splits cleanly: light *quantity* (midday PPFD) drives the carbon
and water fluxes (A, E, iWUE, LUE), while light *quality* (midday R:FR) drives
the regulatory traits (gs, WUE, deltaT).

LUE is the honest exception. Because LUE = A/PPFD, it is a consequence of PPFD
rather than something the environment predicts independently, so its CV
R-squared sits at about zero.

In [ ]:
d = io_physio.attach_environment(physio, growth)
print(f"{len(d):,} leaf records joined to campaign environment "
      f"({d.period_label.nunique()} campaigns)")

r2_rows, importance = [], {}
for v in C.PHYSIO_VARS:
    cv_r2, imp = M.gbm_regression(d, C.PHYSIO_ENV_FEATURES, v)
    importance[v] = imp
    r2_rows.append({"Trait": v, "CV_R2": round(cv_r2, 3),
                    "top_driver": C.FEATURE_LABELS[imp.idxmax()],
                    "top_pct": round(imp.max(), 1)})

r2 = pd.DataFrame(r2_rows)
r2.to_csv(C.TAB_DIR / "Figure_4_physio_env_CV_R2.csv", index=False)

long = [{"Trait": v, "Feature": C.FEATURE_LABELS[f],
         "Category": C.FEATURE_CATEGORY[f], "Importance_pct": round(importance[v][f], 1)}
        for v in C.PHYSIO_VARS for f in C.PHYSIO_ENV_FEATURES]
pd.DataFrame(long).to_csv(C.TAB_DIR / "Figure_4_physio_env_importance.csv", index=False)
print(r2.to_string(index=False))

In [ ]:
PANEL_LABEL = {"A": "A  (net assimilation)", "gs": "gₛ  (stomatal conductance)",
               "E": "E  (transpiration)", "WUE": "WUE", "iWUE": "iWUE",
               "LUE": "LUE", "deltaT": "ΔT (leaf − air)"}
# Gas exchange was measured at midday only, so every light feature is one category.
CATEGORY = {f: ("Light" if C.FEATURE_CATEGORY[f].startswith("Light")
                else C.FEATURE_CATEGORY[f]) for f in C.PHYSIO_ENV_FEATURES}

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.ravel()
for i, v in enumerate(C.PHYSIO_VARS):
    ax = axes[i]
    s = importance[v].sort_values()
    ax.barh(range(len(s)), s.values, alpha=0.92, edgecolor="white",
            color=[P.CATEGORY_COLORS[CATEGORY[f]] for f in s.index])
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels([C.FEATURE_LABELS[f] for f in s.index], fontsize=14)
    for j, val in enumerate(s.values):
        if val > 1:
            ax.text(val + 0.5, j, f"{val:.0f}", va="center", fontsize=13)
    cv = r2.loc[r2.Trait == v, "CV_R2"].iloc[0]
    ax.set_xlabel("Importance (%)", fontsize=16)
    ax.set_xlim(0, max(s.values) * 1.22)
    ax.annotate(f"R² = {cv:.2f}", xy=(0.96, 0.06), xycoords="axes fraction",
                ha="right", fontsize=14,
                bbox=dict(boxstyle="round", fc="white", ec="#999", alpha=0.85))
    ax.set_ylabel(PANEL_LABEL[v], fontsize=15)
    P.despine(ax)
    P.panel(ax, "ABCDEFG"[i], x=-0.02, y=1.05, size=21)

axes[7].axis("off")
axes[7].legend(handles=[mpatches.Patch(color=P.CATEGORY_COLORS[c], label=c)
                        for c in ["Light", "Thermal", "Photoperiod", "Water"]],
               loc="upper center", bbox_to_anchor=(0.5, 1.02), frameon=False,
               fontsize=16, title="Driver type", title_fontsize=17, labelspacing=0.35)
axes[7].text(0.5, 0.12, "Φ is the slope of A vs light\n(one value per group) — compared\n"
                        "by ANCOVA, not modelled here.",
             ha="center", va="center", fontsize=13.5, style="italic",
             transform=axes[7].transAxes)
fig.tight_layout(w_pad=2.5, h_pad=3)
P.save(fig, "Figure_4")

## Key numbers quoted

In [ ]:
print("Top driver and its share, per trait:")
for v in C.PHYSIO_VARS:
    top = importance[v].sort_values(ascending=False).head(2)
    print(f"  {v:7s} " + "  ".join(f"{C.FEATURE_LABELS[f]} {p:.0f}%" for f, p in top.items()))

print("\nApparent quantum yield (slope of A vs PPFD):")
for env, a in aqy.items():
    print(f"  {io_physio.SYSTEM_LABEL[env]:4s} FE {a['F']:.5f}  MA {a['M']:.5f}  "
          f"ANCOVA p = {a['p']:.3f} {S.stars(a['p'])}")

In [ ]:
mo = table_2[table_2.System == "MO"].set_index("Trait")
afs = table_2[table_2.System == "AFS"].set_index("Trait")
assert abs(mo.loc["A", "FE_mean"] - 6.909) < 0.01
assert abs(mo.loc["iWUE", "FE_mean"] - 35.37) < 0.01 and mo.loc["iWUE", "sig"] == "***"
assert abs(afs.loc["LUE", "FE_mean"] - 0.0759) < 0.001 and afs.loc["LUE", "sig"] == "**"
assert abs(r2.set_index("Trait").loc["A", "top_pct"] - 81.0) < 0.5
print("Validated the Table 2 and Figure 4 results used in the manuscript.")